In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import pygeohash as pgh
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from collections import deque
from math import radians, cos, sin, asin, sqrt

import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("dataset")

TRAIN_PATH = BASE_DIR / "train.csv"
TEST_PATH  = BASE_DIR / "test.csv"

In [34]:
# ─────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────

def build_neighbor_map(all_geohashes, loc_demand_map):
    """
    Spatial geometry only - demand values come exclusively from loc_demand_map.
    """
    DELTA = 0.0055
    gh_set = set(all_geohashes)
    neighbor_means = {}
    for gh in all_geohashes:
        lat, lon = pgh.decode(gh)
        neighbors = []
        for dlat, dlon in [(DELTA,0),(-DELTA,0),(0,DELTA),(0,-DELTA),
                           (DELTA,DELTA),(DELTA,-DELTA),(-DELTA,DELTA),(-DELTA,-DELTA)]:
            candidate = pgh.encode(lat+dlat, lon+dlon, precision=6)
            if candidate in gh_set and candidate != gh:
                neighbors.append(candidate)
        if neighbors:
            vals = [v for v in [loc_demand_map.get(n, np.nan) for n in neighbors] if not np.isnan(v)]
            neighbor_means[gh] = np.mean(vals) if vals else np.nan
        else:
            neighbor_means[gh] = np.nan
    return neighbor_means

def haversine(lon1, lat1, lon2, lat2):
    """Calculate the great circle distance in kilometers."""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 
    return c * r

In [35]:
# ─────────────────────────────────────────────────────────────
# 1. LOAD — keep train and test separate from the start
# ─────────────────────────────────────────────────────────────
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

print("Data loaded...")

Data loaded...


**Cyclic Time Features**

To capture the cyclical nature of time features, we can use sine and cosine transformations. This allows the model to understand that, for example, hour 23 and hour 0 are close to each other.

In [36]:
def parse_time(df):
    df = df.copy()
    df['hour']        = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']      = df['timestamp'].str.split(':').str[1].astype(int)
    df['mins_of_day'] = df['hour'] * 60 + df['minute']
    df['time_bucket'] = df['mins_of_day'] // 15
    return df

train_raw = parse_time(train_raw)
test_raw  = parse_time(test_raw)

print("Time features parsed...")

Time features parsed...


In [37]:
# ─────────────────────────────────────────────────────────────
# 2. WITHIN-DAY LAG FEATURES (train only, real values)
# ─────────────────────────────────────────────────────────────
print("Computing within-day lag features...")

train_raw = train_raw.sort_values(['geohash', 'day', 'time_bucket']).reset_index(drop=True)

for lag in [1, 2, 4]:
    train_raw[f'lag_{lag}'] = (
        train_raw
        .groupby(['geohash', 'day'])['demand']
        .shift(lag)
    )

# Exponentially Weighted Mean of past demand values within the same day, grouped by geohash.
# Higher alpha means more weight to recent values, which is desirable for short-term demand forecasting. Shift by 1 to avoid leakage.
train_raw['ewm_demand'] = (
    train_raw
    .groupby(['geohash', 'day'])['demand']
    .transform(
        lambda x: x.ewm(alpha=0.6, adjust=False).mean().shift(1)
    )
)

# Lag differences and ratios to capture trends and relative changes
# Positive lag_diff indicates that demand is increasing, negative indicates decreasing.
# Lag ratios can capture relative changes, but we add a small constant to avoid division by zero.
train_raw['lag_diff_1']  = train_raw['lag_1'] - train_raw['lag_2']
train_raw['lag_diff_2']  = train_raw['lag_2'] - train_raw['lag_4']
# train_raw['lag_ratio_1'] = train_raw['lag_1'] / (train_raw['lag_4'] + 1e-6)

# Introduce noise to lag features for regularization, but only for a random 30% subset of the training data to preserve overall signal quality.
# noise_mask = np.random.rand(len(train_raw)) < 0.30

# train_raw.loc[noise_mask, 'lag_1'] *= (
#     1 + np.random.normal(0, 0.08, noise_mask.sum())
# )

# train_raw.loc[noise_mask, 'lag_2'] *= (
#     1 + np.random.normal(0, 0.08, noise_mask.sum())
# )

# train_raw.loc[noise_mask, 'ewm_demand'] *= (
#     1 + np.random.normal(0, 0.05, noise_mask.sum())
# )

# test lag columns: NaN placeholders, filled during rolling inference
for col in ['lag_1','lag_2','lag_4','ewm_demand','lag_diff_1','lag_diff_2']:
    test_raw[col] = np.nan

print("Lag features computed...")

Computing within-day lag features...
Lag features computed...


In [38]:
# ─────────────────────────────────────────────────────────────
# 3. STATISTICS — computed from train only, merged onto both
# ─────────────────────────────────────────────────────────────
print("Computing train-only statistics...")

# 3a. Geohash × time_bucket demand profile (2D lookup)
gh_ts_profile = (
    train_raw
    .groupby(['geohash', 'time_bucket'])['demand']
    .agg(loc_ts_mean='mean', loc_ts_std='std')
    .reset_index()
)
ts_global = train_raw.groupby('time_bucket')['demand'].mean().rename('ts_global_mean')
n_counts  = train_raw.groupby(['geohash','time_bucket'])['demand'].count().rename('n')

gh_ts_profile = (
    gh_ts_profile
    .join(ts_global, on='time_bucket')
    .join(n_counts,  on=['geohash','time_bucket'])
)
SMOOTH_PROFILE = 25
gh_ts_profile['loc_ts_mean_smooth'] = (
    (gh_ts_profile['loc_ts_mean'] * gh_ts_profile['n'] +
     gh_ts_profile['ts_global_mean'] * SMOOTH_PROFILE)
    / (gh_ts_profile['n'] + SMOOTH_PROFILE)
)
gh_ts_profile['loc_ts_std'] = gh_ts_profile['loc_ts_std'].fillna(0)

print("- Geohash-time_bucket profiles computed...")

# 3b. Location-level stats
loc_stats = train_raw.groupby('geohash')['demand'].agg(
    loc_mean='mean', 
    loc_std='std', 
    # loc_max='max',
    # loc_median='median',
    # loc_p25=lambda x: x.quantile(0.25),
    # loc_p75=lambda x: x.quantile(0.75)
).reset_index()

# 3c. Timestamp-level stats
ts_stats = train_raw.groupby('timestamp')['demand'].agg(
    ts_mean='mean', 
    ts_std='std', 
    # ts_median='median', ts_max='max' 
).reset_index()

# 3d. RoadType × time_bucket interaction
train_raw['RoadType'] = train_raw['RoadType'].fillna('Unknown')
road_ts_stats = (
    train_raw
    .groupby(['RoadType', 'time_bucket'])['demand']
    .agg(road_ts_mean='mean')
    .reset_index()
)

# 3e. Imputation values — derived from TRAIN only
lanes_median = train_raw['NumberofLanes'].median()
temp_median  = train_raw['Temperature'].median()

print(f"- Train-only lanes median: {lanes_median}, temp median: {temp_median:.2f}")

# 3f. demand_yesterday lookup (train only)
lookup_dict = train_raw.set_index(['geohash', 'timestamp', 'day'])['demand'].to_dict()

Computing train-only statistics...
- Geohash-time_bucket profiles computed...
- Train-only lanes median: 2.0, temp median: 16.38


In [39]:
# ─────────────────────────────────────────────────────────────
# 4. COORDINATE EXTRACTION (pure math, no leakage)
# ─────────────────────────────────────────────────────────────
print("Extracting coordinates...")

# All unique geohashes across train + test (needed for neighbor map geometry)
all_geohashes = pd.concat([
    train_raw['geohash'], test_raw['geohash']
]).unique()

coords = {gh: pgh.decode(gh) for gh in all_geohashes}
for df in [train_raw, test_raw]:
    df['lat'] = df['geohash'].map(lambda g: coords[g][0])
    df['lon'] = df['geohash'].map(lambda g: coords[g][1])

print("Coordinates extracted...")

Extracting coordinates...
Coordinates extracted...


In [40]:
# ─────────────────────────────────────────────────────────────
# 5. ENCODERS — fit on TRAIN, transform both
# ─────────────────────────────────────────────────────────────

# DEBATABLE
# le = LabelEncoder()
# le.fit(train_raw['geohash'])

# # Unseen test geohashes get -1 (shouldn't happen with masked data but safe)
# def safe_label_encode(series, encoder):
#     known = set(encoder.classes_)
#     return series.map(lambda g: encoder.transform([g])[0] if g in known else -1)

# train_raw['geohash_cat'] = safe_label_encode(train_raw['geohash'], le)
# test_raw['geohash_cat']  = safe_label_encode(test_raw['geohash'],  le)

# train_raw['geohash_cat'] = train_raw['geohash_cat'].astype('category')
# test_raw['geohash_cat']  = test_raw['geohash_cat'].astype('category')

# DEBATABLE: Busiest location can shift over time
# dist_to_center reference derived from train only
# busiest_gh = train_raw.groupby('geohash')['demand'].mean().idxmax()
# c_lat, c_lon = pgh.decode(busiest_gh)
# for df in [train_raw, test_raw]:
#     df['dist_to_center'] = df.apply(
#         lambda r: haversine(r['lon'], r['lat'], c_lon, c_lat), axis=1)

In [41]:
# KMeans fitted on TRAIN lat/lon only
print("Fitting K-Means on train coordinates only...")

kmeans = KMeans(n_clusters=24, random_state=42, n_init=10)
kmeans.fit(train_raw[['lat', 'lon']])

train_raw['zone_id'] = kmeans.predict(train_raw[['lat', 'lon']])
test_raw['zone_id']  = kmeans.predict(test_raw[['lat', 'lon']])

# Make sure zone_id is treated as categorical for modeling
train_raw['zone_id'] = train_raw['zone_id'].astype('category')
test_raw['zone_id']  = test_raw['zone_id'].astype('category')

# Geohash prefix hierarchy (pure string slicing, no fitting needed)
for df in [train_raw, test_raw]:
    df['gh_prefix4'] = df['geohash'].str[:4].astype('category')
    # df['gh_prefix5'] = df['geohash'].str[:5].astype('category')

Fitting K-Means on train coordinates only...


In [42]:
# ─────────────────────────────────────────────────────────────
# 6. FEATURE APPLICATION PIPELINE
# ─────────────────────────────────────────────────────────────

print("Applying engineered features...")

# 1. MERGE TRAIN-DERIVED GLOBAL STATISTICS
def merge_global_statistics(df):
    df = df.merge(
        loc_stats,
        on='geohash',
        how='left'
    )

    df = df.merge(
        ts_stats,
        on='timestamp',
        how='left'
    )

    df = df.merge(
        road_ts_stats,
        on=['RoadType', 'time_bucket'],
        how='left'
    )

    df = df.merge(
        gh_ts_profile[['geohash', 'time_bucket', 'loc_ts_mean_smooth', 'loc_ts_std']],
        on=['geohash', 'time_bucket'],
        how='left'
    )

    # fallback handling
    df['road_ts_mean'] = (
        df['road_ts_mean']
        .fillna(df['ts_mean'])
    )

    df['loc_ts_mean_smooth'] = (
        df['loc_ts_mean_smooth']
        .fillna(df['ts_mean'])
    )

    df['loc_ts_std'] = (
        df['loc_ts_std']
        .fillna(0)
    )

    return df


# 2. DEMAND YESTERDAY
def add_previous_day_signal(df):
    df['demand_yesterday'] = np.nan

    mask = df['day'] > 48

    df.loc[mask, 'demand_yesterday'] = df[mask].apply(
        lambda r: lookup_dict.get(
            (r['geohash'], r['timestamp'], r['day'] - 1),
            np.nan
        ),
        axis=1
    )

    df['demand_yesterday'] = (
        df['demand_yesterday']
        .fillna(df['loc_ts_mean_smooth'])
    )

    return df


# 3. LAG IMPUTATION
def impute_lag_features(df):
    lag_cols = ['lag_1','lag_2','lag_4','ewm_demand','lag_diff_1','lag_diff_2'] #,'lag_ratio_1']

    for col in lag_cols:
        df[col] = (
            df[col]
            .fillna(df['loc_ts_mean_smooth'])
        )

    df['momentum_ratio'] = df['demand_yesterday']/(df['loc_ts_mean_smooth'] + 1e-6)

    return df


# 4. CYCLICAL TIME FEATURES
def add_time_features(df):
    df['time_sin'] = np.sin(2 * np.pi * df['mins_of_day'] / 1440)
    df['time_cos'] = np.cos(2 * np.pi * df['mins_of_day'] / 1440)

    df['is_rush_hour'] = (
        ((df['hour'] >= 7) & (df['hour'] < 9)) |
        ((df['hour'] >= 17) & (df['hour'] < 19))
    ).astype(int)

    df['is_peak_window'] = (
        (df['hour'] >= 6) &
        (df['hour'] < 20)
    ).astype(int)
    
    return df


# 5. CLEAN CATEGORICALS
def clean_categoricals(df):
    categorical_cols = [
        'RoadType',
        'Weather',
        'LargeVehicles',
        'Landmarks',
        'zone_id',
        'gh_prefix4',
        # 'gh_prefix5'
    ]

    for col in categorical_cols:
        # convert to string first
        df[col] = df[col].astype(str)

        # then fill missing
        df[col] = df[col].fillna('Unknown')

        # finally convert to category
        df[col] = df[col].astype('category')

    return df


# 6. ENVIRONMENT FEATURES
def clean_environment(df):
    df['NumberofLanes'] = (df['NumberofLanes'].fillna(lanes_median))
    df['Temperature'] = (df['Temperature'].fillna(temp_median))

    return df


# 7. MASTER PIPELINE
def apply_features(df):
    df = df.copy()

    df['RoadType'] = (
        df['RoadType']
        .fillna('Unknown')
    )

    df = merge_global_statistics(df)
    df = add_previous_day_signal(df)
    df = impute_lag_features(df)
    df = add_time_features(df)
    df = clean_categoricals(df)
    df = clean_environment(df)

    return df

Applying engineered features...


In [43]:
print("Computing spatial neighbor map...")

loc_demand_map = (
    train_raw
    .groupby('geohash')['demand']
    .mean()
    .to_dict()
)

neighbor_map = build_neighbor_map(
    all_geohashes,
    loc_demand_map
)

# train_raw['neighbor_mean_demand'] = (
#     train_raw['geohash']
#     .map(neighbor_map)
# )

# test_raw['neighbor_mean_demand'] = (
#     test_raw['geohash']
#     .map(neighbor_map)
# )

# global_neighbor_mean = np.mean(
#     list(loc_demand_map.values())
# )

# train_raw['neighbor_mean_demand'] = (
#     train_raw['neighbor_mean_demand']
#     .fillna(train_raw['geohash'].map(loc_demand_map))
#     .fillna(global_neighbor_mean)
# )

# test_raw['neighbor_mean_demand'] = (
#     test_raw['neighbor_mean_demand']
#     .fillna(test_raw['geohash'].map(loc_demand_map))
#     .fillna(global_neighbor_mean)
# )

train_df = apply_features(train_raw)
test_df  = apply_features(test_raw)

FEATURES = [
    'time_sin',
    'time_cos',
    'time_bucket',
    'is_rush_hour',
    'is_peak_window',

    'lat',
    'lon',

    # 'zone_id',
    'gh_prefix4',
    # 'gh_prefix5',

    'loc_mean',
    'loc_std',

    # 'ts_mean',
    # 'ts_std',

    'loc_ts_mean_smooth',
    'loc_ts_std',
    
    'momentum_ratio',

    'road_ts_mean',

    # 'neighbor_mean_demand',

    'demand_yesterday',

    'lag_1',
    'lag_2',
    'lag_4',
    'ewm_demand',

    'lag_diff_1',
    'lag_diff_2',
    # 'lag_ratio_1', 

    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',

    'NumberofLanes',
    'Temperature',
]

X = train_df[FEATURES].copy()

y = np.log1p(train_df['demand'])

print("\nFeature summary")
print("-" * 50)

print(f"Rows: {len(X)}")
print(f"Features: {len(FEATURES)}")

print("\nLag availability:")
print(f"lag_1: {X['lag_1'].notna().mean():.2%}")
print(f"lag_4: {X['lag_4'].notna().mean():.2%}")

print("\nCategorical features:")

cat_cols = X.select_dtypes(include='category').columns.tolist()

print(cat_cols)

Computing spatial neighbor map...

Feature summary
--------------------------------------------------
Rows: 77299
Features: 27

Lag availability:
lag_1: 100.00%
lag_4: 100.00%

Categorical features:
['gh_prefix4', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']


In [44]:
print("Training LightGBM with rolling autoregressive forecasting...")

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.02,
    'max_depth': 6,
    'num_leaves': 32,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.75,
    'bagging_freq': 3,
    'min_data_in_leaf': 64,
    'lambda_l1': 1.0,
    'lambda_l2': 2.0,
    'verbose': -1,
    'seed': 42,
}

# The train setup contains Day 48 (all buckets) and Day 49 (buckets 0-8).
# The test setup evaluates Day 49 (buckets 9-55).
# To prevent empty datasets, we mock a temporal splits:
# Train:
#   - full day 48
# Validate:
#   - available early day 49

pseudo_train_idx = train_df[
    (train_df['day'] == 48) |
    (
        (train_df['day'] == 49) &
        (train_df['time_bucket'] <= 4)
    )
].index

pseudo_valid_idx = train_df[
    (train_df['day'] == 49) &
    (train_df['time_bucket'] > 4)
].index

X = train_df[FEATURES].copy()
y = np.log1p(train_df['demand'])

X_train = X.loc[pseudo_train_idx].copy()
y_train = y.loc[pseudo_train_idx]

X_valid = X.loc[pseudo_valid_idx].copy()
y_valid = y.loc[pseudo_valid_idx]

# Target encoding for geohash
# Helps model learn average demand tendencies per location

SMOOTH_ENC = 35

gh_train = train_df.loc[pseudo_train_idx, 'geohash']

global_mean = y_train.mean()

enc_stats = (
    pd.DataFrame({
        'gh': gh_train.values,
        'target': y_train.values
    })
    .groupby('gh')['target']
    .agg(['mean', 'count'])
)

enc_stats['encoded'] = (
    (
        enc_stats['mean'] * enc_stats['count']
        + global_mean * SMOOTH_ENC
    )
    /
    (enc_stats['count'] + SMOOTH_ENC)
)

gh_enc_map = enc_stats['encoded'].to_dict()

X_train['geohash_target_enc'] = (
    gh_train
    .map(gh_enc_map)
    .fillna(global_mean)
    .values
)

X_valid['geohash_target_enc'] = (
    train_df.loc[pseudo_valid_idx, 'geohash']
    .map(gh_enc_map)
    .fillna(global_mean)
    .values
)

feat_cols = FEATURES + ['geohash_target_enc']

train_ds = lgb.Dataset(
    X_train[feat_cols],
    label=y_train
)

valid_ds = lgb.Dataset(
    X_valid[feat_cols],
    label=y_valid,
    reference=train_ds
)

model = lgb.train(
    params,
    train_ds,
    num_boost_round=2500,
    valid_sets=[train_ds, valid_ds],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=True
        ),
        lgb.log_evaluation(period=100),
    ]
)

Training LightGBM with rolling autoregressive forecasting...
Training until validation scores don't improve for 100 rounds
[100]	training's rmse: 0.0272046	valid_1's rmse: 0.0272098
[200]	training's rmse: 0.0211338	valid_1's rmse: 0.0235159
Early stopping, best iteration is:
[182]	training's rmse: 0.0215056	valid_1's rmse: 0.0234775


In [45]:
# Validation predictions
valid_preds = np.expm1(
    model.predict(X_valid[feat_cols])
)

# Ground truth
valid_truth = np.expm1(y_valid)

# Compute R²
valid_r2 = r2_score(
    valid_truth,
    valid_preds
)

print(f"\nValidation R²: {valid_r2:.4f}")


Validation R²: 0.9590


In [46]:
# ============================================================
# RECURSIVE TEST INFERENCE
# ============================================================

print("\nRunning rolling autoregressive inference...")

day49_known = (
    train_df[train_df['day'] == 49]
    .sort_values(['geohash', 'time_bucket'])
)

day48_tail = (
    train_df[train_df['day'] == 48]
    .sort_values(['geohash', 'time_bucket'])
    .groupby('geohash')
    .tail(4)
    .groupby('geohash')['demand']
    .apply(list)
)

lag_buffer = {}

for gh in test_df['geohash'].unique():
    d49_vals = (
        day49_known[
            day49_known['geohash'] == gh
        ]['demand']
        .tolist()
    )

    d48_vals = list(
        day48_tail.get(gh, [0.05])
    )

    seed = (
        (d48_vals + d49_vals)[-4:]
        if d49_vals
        else d48_vals[-4:]
    )

    lag_buffer[gh] = deque(
        seed,
        maxlen=4
    )

# Add target encoding
test_df['geohash_target_enc'] = (
    test_df['geohash']
    .map(gh_enc_map)
    .fillna(global_mean)
)

# Chronological order
test_df = (
    test_df
    .sort_values(['time_bucket', 'geohash'])
    .reset_index(drop=True)
)

# Helper
def get_test_lag(gh, n):
    vals = list(
        lag_buffer.get(
            gh,
            deque([0.05], maxlen=4)
        )
    )

    if len(vals) >= n:
        return vals[-n]

    return vals[0]

test_predictions = {}

for tb, group in test_df.groupby('time_bucket', sort=True):
    rows = group.copy()

    rows['lag_1'] = rows['geohash'].map(
        lambda g: get_test_lag(g, 1)
    )

    rows['lag_2'] = rows['geohash'].map(
        lambda g: get_test_lag(g, 2)
    )

    rows['lag_4'] = rows['geohash'].map(
        lambda g: get_test_lag(g, 4)
    )

    rows['ewm_demand'] = rows['geohash'].map(
        lambda g:
            sum(
                get_test_lag(g, n) * 0.7**(n - 1)
                for n in range(1, 5)
            )
            /
            sum(0.7**i for i in range(4))
    )

    rows['lag_diff_1'] = (
        rows['lag_1'] - rows['lag_2']
    )

    rows['lag_diff_2'] = (
        rows['lag_2'] - rows['lag_4']
    )

    rows['geohash_target_enc'] = (
        rows['geohash']
        .map(gh_enc_map)
        .fillna(global_mean)
    )

    rows['forecast_step'] = tb - 9

    step_preds = np.expm1(
        model.predict(rows[feat_cols])
    )

    step_preds = np.maximum(step_preds, 0)

    # Stabilization
    step_preds = (
        0.93 * step_preds
        +
        0.07 * rows['loc_ts_mean_smooth'].values
    )

    for i, (_, row) in enumerate(rows.iterrows()):

        pred = step_preds[i]

        # Smooth recursive memory
        prev = get_test_lag(row['geohash'], 1)

        smoothed_pred = (
            0.75 * pred
            +
            0.25 * prev
        )

        test_predictions[row['Index']] = pred

        lag_buffer[row['geohash']].append(
            smoothed_pred
        )

print("Test inference complete.")


Running rolling autoregressive inference...
Test inference complete.


In [47]:
# Build submission
print("\nBuilding submission file...")

submission = pd.read_csv(TEST_PATH)[['Index']].copy()

submission['demand'] = (
    submission['Index']
    .map(test_predictions)
)


# Safety fallback
fallback_value = np.expm1(y).mean()

submission['demand'] = (
    submission['demand']
    .fillna(fallback_value)
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("Saved submission.csv")
print(submission.head())


Building submission file...
Saved submission.csv
   Index    demand
0      0  0.052692
1      1  0.030336
2      2  0.034712
3      3  0.028810
4      4  0.070843


In [48]:
importance = pd.DataFrame({
    'feature': feat_cols,
    'importance': model.feature_importance()
})

print(
    importance.sort_values(
        'importance',
        ascending=False
    ).head(20)
)

               feature  importance
18          ewm_demand         645
10  loc_ts_mean_smooth         639
8             loc_mean         487
15               lag_1         485
12      momentum_ratio         395
13        road_ts_mean         357
0             time_sin         355
14    demand_yesterday         355
1             time_cos         338
2          time_bucket         268
9              loc_std         208
27  geohash_target_enc         208
19          lag_diff_1         129
5                  lat         126
11          loc_ts_std         109
16               lag_2          51
4       is_peak_window          39
21            RoadType          32
20          lag_diff_2          31
6                  lon          29
